In [15]:
import torch
import pandas as pd

# List of propensity models
models = ['logistic', 'truncated_logistic', 'truncated_adv']

# Create lists to store results
model_names = []
rmse_linear_list = []
sd_linear_list = []
rmse_riesz_list = []
sd_riesz_list = []
mea_linear_list = []
mea_riesz_list = []

# Read and compute RMSE and SD for each model
for model in models:
    pred_theta = torch.load(f"results/{model}_pred_theta.pt")
    pred_sig = torch.load(f"results/{model}_pred_sig.pt")
    ATT = torch.load(f"results/{model}_ATT_calculations.pt")["ATT"]

    rmse_linear = torch.sqrt(torch.mean((pred_theta[:, 1] - ATT)**2)).item()
    mea_linear = torch.mean(torch.abs((pred_theta[:, 1] - ATT)**2)).item()
    sd_linear = torch.mean(pred_sig[:, 1]).item()
    rmse_riesz = torch.sqrt(torch.mean((pred_theta[:, 3] - ATT)**2)).item()
    sd_riesz = torch.mean(pred_sig[:, 3]).item()
    mea_riesz  = torch.mean(torch.abs((pred_theta[:, 3] - ATT)**2)).item()

    model_names.append(model)
    rmse_linear_list.append(rmse_linear)
    sd_linear_list.append(sd_linear)
    rmse_riesz_list.append(rmse_riesz)
    sd_riesz_list.append(sd_riesz)
    mea_linear_list.append(mea_linear)
    mea_riesz_list.append(mea_riesz)
results_df = pd.DataFrame({
    'Model': model_names,
    'RMSE (Linear)': rmse_linear_list,
    "MAE (Linear)": mea_linear_list,
    'SD (Linear)': sd_linear_list,
    'RMSE (Dynamic Riesz)': rmse_riesz_list,
            "MAE (Dynamic Riesz)": mea_riesz_list,

    'SD (Dynamic Riesz)': sd_riesz_list,
})

results_df

,Model,RMSE (Linear),MAE (Linear),SD (Linear),RMSE (Dynamic Riesz),MAE (Dynamic Riesz),SD (Dynamic Riesz)
0,logistic,0.200696,0.040279,3.906547,0.185914,0.034564,6.803707
1,truncated_logistic,0.191084,0.036513,3.906988,0.184830,0.034162,6.802573
2,truncated_adv,0.195961,0.038401,3.907142,0.184397,0.034002,6.804862


In [ ]:
def cov(thetas, sigmas, N):

    ub = thetas + 1.96 * sigmas/ torch.sqrt(torch.tensor(N))
    lb = thetas - 1.96 * sigmas / torch.sqrt(torch.tensor(N))
    coverage = torch.mean( ( (ATT >= lb) & (ATT <= ub) ).float() , 0 )
    interval_length = torch.mean(ub - lb,0)
    return {"Coverage":coverage,
            "interval_length": interval_length}


In [17]:
coverage

tensor(0.8100)

In [18]:
interval_length

tensor(0.4843)